## Rectangular Superposition

by: Brett Mattas
<br>date: 5/17/2016

### Description

Explore rectangular superposition explained in section XX


### Included:

1. Item
2. Item
3. Item

### Section 3.4 Loading on a Rectangular Area

![rectangular distribution with coordinates](./images/rectangular_distribution.png)

The pressures defined underneath a corner and assuming a poisson's ratio of 0.5 are:

Equation 3.18a:



$$

\sigma_z
= \frac{p}{2\pi}
\left[
\tan^{-1}\left(\frac{lb}{zR_3}\right)
+
\frac{lbz}{R_3}
\left(\frac{1}{R_1^{2}}+\frac{1}{R_2^2}\right)
\right]

\\[2em]
where:\\


R_1 = \sqrt{l^{2} + z^{2}}\\
R_2 = \sqrt{b^{2} + z^{2}}\\
R_3 = \sqrt{l^{2} + b^{2} + z^{2}}\\

\\[1em]
and:\\

q = uniform \: vertical \: pressure\\
l = length \: in \: the \: y \: direction\\
b = width \: in \: the \: x \: direction\\
z = depth \: below \: the \: loaded \: corner\\

$$



Forces in the other directions will be included in the module, but for this notebook I am only exploring how to do the rectangular superposition and will focus on vertical pressure.

In [1]:
import math
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from buried_structures import Point_3d, Origin
import xarray as xr
print("Finished imports")


Finished imports


In [2]:
def rectangular_stress(p: float, l: float, b: float, z: float) -> float:
    # Vertical pressure under the corner of a rectangular applied load

    # These quick exits are useful later when using superposition
    # to get pressure under than directly under a corner
    if l == 0 or b == 0:
        return 0
    if p == 0:
        return 0
    
    if l < 0:
        raise ValueError("l needs to be a positive number")
    if b < 0:
        raise ValueError("b needs to be a positive number")
    if p < 0:
        raise ValueError("p needs to be a positive number")
    if z < 0:
        raise ValueError("z needs to be a positive value or an error")
    
    z = z if z != 0 else 0.000001 # Avoids a singularity

    R1 = math.sqrt(l**2 + z**2)
    R2 = math.sqrt(b**2 + z**2)
    R3 = math.sqrt(l**2 + b**2 + z**2)

    if R1 == 0:
        raise ZeroDivisionError("R1 cannot be 0")
    if R2 == 0:
        raise ZeroDivisionError("R2 cannot be 0")
    if R3 == 0:
        raise ZeroDivisionError("R3 cannot be 0")

    # Break up formula for debugging
    front = p / (2*math.pi)
    first = math.atan((l*b) / (z*R3))
    second = l*b*z/R3
    third = 1/(R1**2) + 1/(R2**2)
    return front * (first + second * third)

In [3]:
# Now let's plot the pressure under a corner
z_values = np.linspace(0, 21, 22)
stress_values = [rectangular_stress(1.0, 10, 20, z) for z in z_values]

fig = px.line(
    x=stress_values,
    y=z_values,
    labels={'x': 'Vertical stress', 'y': 'Depth'},
    title='Rectangular stress vs depth (l=10, b=20, p=1)'
)
fig.update_yaxes(autorange="reversed")
fig.show()

First observation is that although z = 0 would result in a divide by 0 problem, the pressure doesn't diverge as you get closer and closer to z=0. So we can just build in minimum value for z in the formula and allow an input of z = 0.

## Superposition

Poulos & Davis section 1.7 demonstrate how to find stresses from points under a rectangular loading other than a corner.:

![two rectangles with points either inside or outside and numbered boxes to all four corners](./images/rectangle_superposition.png)

For a pont inside the rectangular load, you simply add up the effect of rectangles 1 - 4.

In [4]:
def interior_stress(p: float, l: float, b: float,
                   point: Point_3d) -> float:
    # Return vertical stress of a point inside the rectangular
    # load. Assume load is centered on 0, 0 for now.

    x, y, z = point.coordinates

    if not (-b/2 <= x <= b/2):
        raise ValueError("x needs to be between -b/2 & b/2")
    if not (-l/2 <= y <= l/2):
        raise ValueError("y needs to be between -l/2 & l/2")

    flippers = [[-1, -1], [-1, 1], [1, 1], [1, -1]]

    ret = 0

    for flip in flippers:
        corner = Point_3d(flip[0] * b/2, flip[1] * l/2)
        length, width = corner.dx(point), corner.dy(point)
        # print(f"{length=}, {width=}")
        # print(corner)
        ret += rectangular_stress(p, 
                                  corner.dx(point),
                                  corner.dy(point),
                                  z)


    return ret

point = Point_3d(10, 5, 1)
print(interior_stress(1, 10, 20, point))


0.24988858964973337


With this, we can now do some fun things like plot the stress at the center and edges compared with a corner:

In [5]:
l = 10 # y-direction
b = 20 # b-direction
p = 1

z_values = np.linspace(0, 21, 22)

# Recalculate here in case change l, b, or p
corner_stress = [rectangular_stress(p, l, b, z) for z in z_values]

center_points = [Point_3d(0, 0, z) for z in z_values]
center_stress = [interior_stress(p, l, b, point) for point in center_points]

# Values on the x=l/2  edge
length_edge_points = [Point_3d(b/2, 0, z) for z in z_values]
length_edge_stress = [interior_stress(p, l, b, point) for point in length_edge_points]


# Values on the y=b/2  edge
width_edge_points = [Point_3d(0, l/2, z) for z in z_values]
width_edge_stress = [interior_stress(p, l, b, point) for point in width_edge_points]

# Test that corner points result in the same values
corner_test_points = [Point_3d(b/2, l/2, z) for z in z_values]
corner_test_stress = [interior_stress(p, l, b, point) for point in corner_test_points]



In [6]:

# Now let's plot the pressure under a corner
stress_values = [corner_stress,
                 center_stress,
                 length_edge_stress,
                 width_edge_stress]
names = ["Corner", "Center", "Length Edge", "Width Edge"]

test = False
if test:
    stress_values.append(corner_test_stress)
    names.append("Corner Test")

# for stress in stress_values:
#     print("\n--------")
#     print(stress)
#     print("--------")

fig = px.line(
    x=stress_values,
    y=z_values,
    labels={'x': 'Vertical stress', 'y': 'Depth'},
    title='Rectangular stress vs depth (l=10, b=20, p=1)'
)
fig.update_yaxes(autorange="reversed")

for i, name in enumerate(names):
    fig.data[i].name = name

fig.show()

# Exterior points:

As shown in the diagrams, exterior points can use superposition too. Poulos & Davis include the case where a point is outside a corner, but don't include a case where it is within one edge.

![two rectangles with points either inside or outside and numbered boxes to all four corners](./images/rectangle_superposition.png)

### Within an Edge

Result is the sum of the two bigger rectangles - two smaller rectangles

# !!!!! ADD GRAPHIC !!!!!

### Outside corner

Result is The furthest corner - 2 intermediate corners + closest corner.

The closest corner (4 in the diagram) has to be added in because it is subtracted off twice.

In [7]:
def exterior_stress(p: float, l: float, b: float,
                   point: Point_3d) -> float:
    # Return vertical stress of a point inside the rectangular
    # load. Assume load is centered on 0, 0 for now.

    x, y, z = point.coordinates
    # print(f"{x=}, {y=}, {z=}")

    if  (-l/2 <= y <= l/2) and (-b/2 <= x <= b/2):
        raise ValueError("Point needs to be outside the rectangle")
    
    # Get stress from four rectangles
    flippers = [[-1, -1], [-1, 1], [1, 1], [1, -1]]
    corners = [Point_3d(flip[0] * b/2, flip[1] * l/2) for flip in flippers]
    stresses = []
    for corner in corners:
        # length, width = corner.dx(point), corner.dy(point)
        # print(f"{length=}, {width=}")
        # print(corner)
        # print(f"corner = {corner}")
        stresses.append(
            rectangular_stress(p, 
                               corner.dx(point),
                               corner.dy(point),
                               z)
        )

    # Determine if within or outside an edge
    within_x = -b/2 <= x <= b/2
    within_y = -l/2 <= y <= l/2
    is_within = within_x or within_y

    # print(f"Exterior {is_within=}")

    

    if not is_within:
        # Closest - Middle 2 + Furthest
        # Sort in ascending order by distance
        distances = [corner.distance(point) for corner in corners]
        # print(f"{distances=}")

        sorted_stresses = [val for _, val in sorted(zip(distances, stresses))]
        # print(f"{stresses=}")
        # print(f"{sorted_stresses=}")
        ret = sorted_stresses[0] - sorted_stresses[1] - sorted_stresses[2] + sorted_stresses[3]
        # -Closest 2 + Further 2
    elif within_x:
        #!!!! Find which two corners have closer y dimensions, subtract those
        flippers = [1, -1, -1, 1] if point.y() > 0 else [-1, 1, 1, -1]
        ret = 0

        for i, flip in enumerate(flippers):
            ret += flip * stresses[i]

    elif within_y:

        flippers = [1, 1, -1, -1] if point.x() > 0 else [-1, -1, 1, 1]
        ret = 0

        for i, flip in enumerate(flippers):
            ret += flip * stresses[i]
    else:
        raise NotImplementedError("Shouldn't get here!!!")
        


    return ret
point = Point_3d(10, 12, 5)
print(point)
print(exterior_stress(p, l, b, point))
print('--------')
point = Point_3d(11, 12, 5)
print(point)
print(exterior_stress(p, l, b, point))

x=10.0, y=12.0, z=5.0
0.02047676886277819
--------
x=11.0, y=12.0, z=5.0
0.017620624608418345


In [8]:
# Plot this compared with the others
length_1_edge_points = [Point_3d(b, 0, z) for z in z_values]
print(f"Length 1: {length_1_edge_points[0]}")
print(f"{p=}, {l=}, {b=}")
length_1_edge_stress = [exterior_stress(p, l, b, point) for point in length_1_edge_points]

length_1_5_edge_points = [Point_3d(1.5*b, 0, z) for z in z_values]
length_1_5_edge_stress = [exterior_stress(p, l, b, point) for point in length_1_5_edge_points]

length_2_edge_points = [Point_3d(2*b, 0, z) for z in z_values]
length_2_edge_stress = [exterior_stress(p, l, b, point) for point in length_2_edge_points]


width_1_edge_points = [Point_3d(0, l, z) for z in z_values]
width_1_edge_stress = [exterior_stress(p, l, b, point) for point in width_1_edge_points]

width_1_5_edge_points = [Point_3d(0, 1.5*l, z) for z in z_values]
width_1_5_edge_stress = [exterior_stress(p, l, b, point) for point in width_1_5_edge_points]

width_2_edge_points = [Point_3d(0, 2*l, z) for z in z_values]
width_2_edge_stress = [exterior_stress(p, l, b, point) for point in width_2_edge_points]

# Then combine all ito one function

Length 1: x=20.0, y=0.0, z=0.0
p=1, l=10, b=20


In [9]:
# Now let's plot the pressure under a corner
stress_values = [corner_stress,
                 center_stress,
                 length_edge_stress,
                 width_edge_stress,
                 length_1_edge_stress,
                 length_1_5_edge_stress,
                 length_2_edge_stress,
                 width_1_edge_stress,
                 width_1_edge_stress,
                 width_2_edge_stress,
]
names = ["Corner", "Center", "Length Edge", "Width Edge", "1x Length", "1.5x Length", "2x Length",
         "1x Width", "1.5x Width", "2x Width"]

fig = px.line(
    x=stress_values,
    y=z_values,
    labels={'x': 'Vertical stress', 'y': 'Depth'},
    title='Rectangular stress vs depth (l=10, b=20, p=1)'
)
fig.update_yaxes(autorange="reversed")

for i, name in enumerate(names):
    fig.data[i].name = name

fig.show()

In [10]:
def rectangular_combined_stress(p: float, l: float, b: float,
                   point: Point_3d) -> float:
    # Combine the interior and exterior stresses
    
    if (-b/2 <= point.x() <= b/2) and (-l/2 <= point.y() <= l/2):
        # print("Interior")
        return interior_stress(p, l, b, point)
    # print("Exterior")
    return exterior_stress(p, l, b, point)

print(f"{l=}, {b=}")
point = Point_3d(10, 12, 5)
print(rectangular_combined_stress(p, l, b, point))
point = Point_3d(11, 12, 5)
print(rectangular_combined_stress(p, l, b, point))

l=10, b=20
0.02047676886277819
0.017620624608418345


In [11]:
def find_extreme(iterable, func=max) -> float|None:
    """
    Recursively find the min or max in nested iterables.
    :param iterable: The nested collection (list, tuple, etc.)
    :param func: The built-in min or max function
    """
    flat_elements = []
    
    for item in iterable:
        # Check if the item is a nested iterable (excluding strings if desired)
        if isinstance(item, (list, tuple, set)):
            # Recursive call to drill into the nested structure
            inner_extreme = find_extreme(item, func)
            if inner_extreme is not None:
                flat_elements.append(inner_extreme)
        else:
            # Base case: item is a single value (int, float, etc.)
            flat_elements.append(item)
            
    return func(flat_elements) if flat_elements else None

# Example usage:
nested_data = [1, [5, [10, -2]], 8, [0]]
print(f"Maximum: {find_extreme(nested_data, max)}") # Output: 10
print(f"Minimum: {find_extreme(nested_data, min)}") # Output: -2

Maximum: 10
Minimum: -2


In [12]:
# Taking a slice at a constant y plane.

# Setup linspaces. Set more to make good contrast images
x_arr = np.linspace(-20, 20, 641)
z_arr = np.linspace(0, 20, 321)
y = 12

# Generate matching dummy pressure data (5x4 array)
# pressure_data = np.random.uniform(14.7, 30.0, size=(5, 4))
pressure_data = []

for z in z_arr:
    new_row = []
    for x in x_arr:
        point = Point_3d(x, y, z)
        # print(point)
        new_row.append(rectangular_combined_stress(p, l, b, point))
    
    pressure_data.append(new_row)

# Create the DataArray
pressure_da = xr.DataArray(
    data=pressure_data,
    dims=("z (in)", "x (in)"),
    coords={
        "z (in)": ("z (in)", z_arr, {"units": "inches"}),
        "x (in)": ("x (in)", x_arr, {"units": "inches"}),
    },
    name="pressure",
    attrs={"units": "ksi"}
)


fig = px.imshow(pressure_da, color_continuous_scale='Turbo', 
                aspect='equal',
                title=f"Pressure at y = {y}")
fig.show()

In [13]:
# Take a slice at a constant x plane:

# Setup linspaces. Set more to make good contrast images

y_arr = np.linspace(-20, 20, 641)
z_arr = np.linspace(0, 20, 321)

x = 0

# Generate matching dummy pressure data (5x4 array)
# pressure_data = np.random.uniform(14.7, 30.0, size=(5, 4))
pressure_data = []



for z in z_arr:
    new_row = []
    for y in y_arr:
        point = Point_3d(x, y, z)
        # print(point)
        new_row.append(rectangular_combined_stress(p, l, b, point))
    
    pressure_data.append(new_row)



pressure_da = xr.DataArray(
    data=pressure_data,
    dims=("z (in)", "y (in)"),
    coords={
        "z (in)": ("z (in)", z_arr, {"units": "inches"}),
        "y (in)": ("y (in)", y_arr, {"units": "inches"}),
    },
    name="pressure",
    attrs={"units": "ksi"}
)

# print(pressure_da)

fig = px.imshow(pressure_da, color_continuous_scale='Turbo', 
                aspect='equal',
                title=f"Pressure at x = {x}")
fig.show()

In [14]:
X, Y, Z = np.mgrid[-40:40:40j, -20:20:40j, 0:40:40j]
x, y, z = X.flatten(), Y.flatten(), Z.flatten()

values = [rectangular_combined_stress(p, l, b, Point_3d(x=ix, y=iy, z=iz)) for ix, iy, iz in zip(x, y, z)]
# for value in values:
#     print(value)

isomin, isomax = find_extreme(values, min), find_extreme(values, max)
print(f"{isomin=}, {isomax=}")
fig = go.Figure(data=go.Volume(
    x=x,
    y=y,
    z=z,
    value=values,
    isomin=(isomin + (isomax - isomin)*0.01),
    isomax=isomax,
    opacity=0.2, # needs to be small to see through all surfaces
    surface_count=25, # needs to be a large number for good volume rendering,
    colorscale='Turbo'
    ))


fig.update_layout(title=f"Rectangular Pressure",
                  width = 800, height=600,
                  scene=dict(
                      xaxis_title="x (in)",
                      yaxis_title = "y (in)",
                      zaxis_title = "z (in)",
                      zaxis=dict(autorange="reversed")
                    )
                  )
fig.show()

isomin=np.float64(-1.1102230246251565e-16), isomax=np.float64(1.0000000000000002)
